In [1]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             roc_curve, auc)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Utilities
import os
import sys
import warnings
import joblib
import json
from pathlib import Path
from datetime import datetime

In [2]:
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [3]:
df = pd.read_csv('../data/processed/processed_reviews.csv')

In [4]:
print(f"\nTotal samples: {len(df)}")
print(f"\nSentiment distribution:")
sentiment_dist = df['sentiment'].value_counts()
for sentiment, count in sentiment_dist.items():
    print(f"   {sentiment}: {count} ({count/len(df)*100:.1f}%)")

print(f"\nLanguages present: {df['language'].nunique()}")
print(f" Categories: {df['product_category'].nunique()}")
print(f" Countries: {df['country'].nunique()}")


Total samples: 30000

Sentiment distribution:
   positive: 10000 (33.3%)
   neutral: 10000 (33.3%)
   negative: 10000 (33.3%)

Languages present: 10
 Categories: 8
 Countries: 14


In [5]:
sample_reviews = df[['sentiment', 'processed_review']].sample(5, random_state=42)
for idx, row in sample_reviews.iterrows():
    print(f"\n{row['sentiment'].upper()}: {row['processed_review'][:150]}...")


POSITIVE: really impressed build quality would buy...

NEGATIVE: mauvaise qualit ne ressemble pa du tout aux photo...

NEGATIVE: produit cass la rception service client inexistant...

NEGATIVE: shipping took three week item damaged star...

NEUTRAL: geht erfllt seinen zweck ausreichend...


In [6]:
X = df['processed_review']
y = df['sentiment']

In [7]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [8]:
joblib.dump(label_encoder, '../artifacts/encoders/target_encoder.pkl')

['../artifacts/encoders/target_encoder.pkl']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded,  # Maintain class distribution
    shuffle=True
)

In [10]:
print('Training set')
train_dist = pd.Series(y_train).value_counts().sort_index()
for i, count in train_dist.items():
    class_name = label_encoder.inverse_transform([i])[0]
    print(f"   {class_name}: {count} ({count/len(y_train)*100:.1f}%)")

print('Testing set')
test_dist = pd.Series(y_test).value_counts().sort_index()
for i, count in test_dist.items():
    class_name = label_encoder.inverse_transform([i])[0]
    print(f"   {class_name}: {count} ({count/len(y_test)*100:.1f}%)")

Training set
   negative: 8000 (33.3%)
   neutral: 8000 (33.3%)
   positive: 8000 (33.3%)
Testing set
   negative: 2000 (33.3%)
   neutral: 2000 (33.3%)
   positive: 2000 (33.3%)


In [11]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,        # Limit to top 5000 features
    min_df=2,                  # Ignore terms that appear in less than 2 documents
    max_df=0.8,                # Ignore terms that appear in more than 80% of documents
    ngram_range=(1, 2),        # Use unigrams and bigrams
    sublinear_tf=True,         # Use 1+log(tf) scaling
    strip_accents='unicode',
    lowercase=True,
    stop_words='english'       # Remove English stopwords
)

In [12]:
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [13]:
feature_names = tfidf_vectorizer.get_feature_names_out()
print(feature_names[:20])

['aber' 'aber erfllt' 'aber nichts' 'aber produkt' 'aber unremarkabel'
 'absolument' 'absolutely' 'absolutely love' 'absoluter'
 'absoluter betrug' 'absoluto' 'absoluto lo' 'acceptable'
 'acceptable price' 'acceptable quality' 'aceptable' 'aceptable para'
 'achat' 'achat je' 'achat le']


In [14]:
# Create the directory if it doesn't exist
os.makedirs('../artifacts/tokenizers', exist_ok=True)

joblib.dump(tfidf_vectorizer, '../artifacts/tokenizers/tfidf_vectorizer.pkl')

['../artifacts/tokenizers/tfidf_vectorizer.pkl']

In [15]:
print("\n📊 TF-IDF Statistics:")
print(f"   Mean non-zero features per document: {X_train_tfidf.getnnz(axis=1).mean():.1f}")
print(f"   Sparsity: {100 * (1 - X_train_tfidf.getnnz() / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])):.2f}%")


📊 TF-IDF Statistics:
   Mean non-zero features per document: 11.2
   Sparsity: 99.16%


In [16]:
nb_results = {}

In [17]:
nb_models = {
    'MultinomialNB': MultinomialNB(),
}

In [18]:
for model_name, model in nb_models.items():
    print(f"\n🔍 Training {model_name}...")
    
    # Train the model
    model.fit(X_train_tfidf, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    nb_results[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'predictions': y_pred
    }
    
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1-Score (macro): {f1_macro:.4f}")
    print(f"   F1-Score (weighted): {f1_weighted:.4f}")


🔍 Training MultinomialNB...
   Accuracy: 1.0000
   F1-Score (macro): 1.0000
   F1-Score (weighted): 1.0000


In [19]:
best_nb_name = max(nb_results, key=lambda x: nb_results[x]['f1_macro'])
best_nb_model = nb_results[best_nb_name]['model']
best_nb_pred = nb_results[best_nb_name]['predictions']

print(f"\n🏆 Best Naive Bayes model: {best_nb_name}")
print(f"   F1-Score (macro): {nb_results[best_nb_name]['f1_macro']:.4f}")


🏆 Best Naive Bayes model: MultinomialNB
   F1-Score (macro): 1.0000


In [20]:
print(classification_report(y_test, best_nb_pred, target_names=label_encoder.classes_))

# Confusion Matrix
cm_nb = confusion_matrix(y_test, best_nb_pred)

fig = px.imshow(
    cm_nb,
    x=label_encoder.classes_,
    y=label_encoder.classes_,
    text_auto=True,
    color_continuous_scale='Blues',
    title=f'Confusion Matrix - {best_nb_name}'
)
fig.update_layout(height=500, width=500)
fig.show()

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      2000
     neutral       1.00      1.00      1.00      2000
    positive       1.00      1.00      1.00      2000

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



In [21]:
# Create the directory if it doesn't exist
os.makedirs('../artifacts/models', exist_ok=True)

joblib.dump(best_nb_model, '../artifacts/models/naive_bayes_model.pkl')

['../artifacts/models/naive_bayes_model.pkl']

In [22]:
nb_results_summary = {
    'model_name': best_nb_name,
    'accuracy': nb_results[best_nb_name]['accuracy'],
    'f1_macro': nb_results[best_nb_name]['f1_macro'],
    'f1_weighted': nb_results[best_nb_name]['f1_weighted'],
    'confusion_matrix': cm_nb.tolist()
}

# Create the directory if it doesn't exist
os.makedirs('../artifacts/metrics', exist_ok=True)
with open('../artifacts/metrics/naive_bayes_metrics.json', 'w') as f:
    json.dump(nb_results_summary, f, indent=4)

In [23]:
lr_initial = LogisticRegression(
    random_state=42,
    max_iter=1000,
    n_jobs=-1
)

lr_initial.fit(X_train_tfidf, y_train)
y_pred_lr_initial = lr_initial.predict(X_test_tfidf)

In [24]:
accuracy_lr = accuracy_score(y_test, y_pred_lr_initial)
f1_macro_lr = f1_score(y_test, y_pred_lr_initial, average='macro')
f1_weighted_lr = f1_score(y_test, y_pred_lr_initial, average='weighted')

In [25]:
print(f"Accuracy: {accuracy_lr:.4f}")
print(f"F1-Score (macro): {f1_macro_lr:.4f}")
print(f"F1-Score (weighted): {f1_weighted_lr:.4f}")

Accuracy: 1.0000
F1-Score (macro): 1.0000
F1-Score (weighted): 1.0000


In [26]:
print(classification_report(y_test, y_pred_lr_initial, target_names=label_encoder.classes_))

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00      2000
     neutral       1.00      1.00      1.00      2000
    positive       1.00      1.00      1.00      2000

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



In [27]:
joblib.dump(lr_initial, '../artifacts/models/logistic_regression_model.pkl')

['../artifacts/models/logistic_regression_model.pkl']